# Travel Time Matrix

This notebook shows how to compute many-to-many network travel times with Graphways. A travel time matrix answers questions like: "How long does it take to get from each origin to each destination over the street network?"

The example uses the small OSM fixture included in this repository, so it runs offline and finishes quickly.

In [ ]:
from pathlib import Path

import graphways as gw

## Load a Local Graph

`SpatialGraph` is reusable. Build it once, then run repeated matrix, route, reachability, and isochrone queries against the same in-memory graph.

In [ ]:
fixture = Path("../tests/fixtures/tiny_map.osm")
if not fixture.exists():
    fixture = Path("tests/fixtures/tiny_map.osm")

xml = fixture.read_text(encoding="utf-8")
graph = gw.SpatialGraph.from_osm(xml, network="walk", retain_all=True)

graph.node_count(), graph.edge_count()

## Define Origins and Destinations

Coordinates are always `(lat, lon)`. Matrix rows correspond to origins and columns correspond to destinations.

In [ ]:
origins = {
    "Block A": (48.0000, 11.0000),
    "Block B": (48.0010, 11.0000),
    "Block C": (48.0020, 11.0010),
}

destinations = {
    "Clinic": (48.0020, 11.0000),
    "Library": (48.0030, 11.0000),
}

origin_names = list(origins)
destination_names = list(destinations)
origin_coords = [origins[name] for name in origin_names]
destination_coords = [destinations[name] for name in destination_names]

## Compute the Matrix

`durations_s` is a list of rows. Each value is travel time in seconds, or `None` when the destination is unreachable within the requested cutoff. `max_minutes` is optional; omit it for an unbounded shortest-travel-time matrix.

In [ ]:
matrix = graph.travel_time_matrix(
    origins=origin_coords,
    destinations=destination_coords,
    max_snap_m=100,
    max_minutes=10,
)

matrix

In [ ]:
def fmt_minutes(seconds):
    if seconds is None:
        return None
    return round(seconds / 60, 1)


table = []
for origin_name, row in zip(origin_names, matrix.durations_s):
    record = {"origin": origin_name}
    for destination_name, seconds in zip(destination_names, row):
        record[f"{destination_name} min"] = fmt_minutes(seconds)
    table.append(record)

table

## Nearest Destination by Travel Time

Once the matrix is computed, downstream analysis is ordinary Python: find the fastest destination, calculate coverage counts, or join the values back to GeoDataFrames.

In [ ]:
nearest = []
for origin_name, row in zip(origin_names, matrix.durations_s):
    reachable = [
        (destination_name, seconds)
        for destination_name, seconds in zip(destination_names, row)
        if seconds is not None
    ]
    if reachable:
        destination_name, seconds = min(reachable, key=lambda item: item[1])
        nearest.append(
            {
                "origin": origin_name,
                "nearest_destination": destination_name,
                "travel_time_min": round(seconds / 60, 1),
            }
        )
    else:
        nearest.append(
            {
                "origin": origin_name,
                "nearest_destination": None,
                "travel_time_min": None,
            }
        )

nearest

## Coverage Within a Time Cutoff

A matrix also makes coverage analysis straightforward. Here, each origin is considered covered if at least one destination is reachable within five minutes.

In [ ]:
cutoff_seconds = 5 * 60

coverage = []
for origin_name, row in zip(origin_names, matrix.durations_s):
    reachable_count = sum(
        seconds is not None and seconds <= cutoff_seconds
        for seconds in row
    )
    coverage.append(
        {
            "origin": origin_name,
            "destinations_within_5_min": reachable_count,
            "covered": reachable_count > 0,
        }
    )

coverage

## Snap Diagnostics

Matrix results include snap diagnostics for every requested origin and destination, matching the route API's behavior.

In [ ]:
{
    "origin_snaps": [snap.as_dict() if snap else None for snap in matrix.origin_snaps],
    "destination_snaps": [snap.as_dict() if snap else None for snap in matrix.destination_snaps],
}